# Open-source LLMs przez OpenRouter — zero-shot i few-shot na LIAR

Testujemy 4 darmowe modele LLM przez OpenRouter:
- `openai/gpt-oss-20b:free`
- `google/gemma-4-31b-it:free`
- `cognitivecomputations/dolphin-mistral-24b-venice-edition:free`
- `meta-llama/llama-3.3-70b-instruct:free`

Pobieramy przygotowany dataset z Drive:

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

CACHE_DIR = '/content/drive/MyDrive/fakenews_cache'
TOK_DIR   = os.path.join(CACHE_DIR, 'tokenized')

print('TOK_DIR:', os.listdir(TOK_DIR))

In [ ]:
%pip install -q requests tqdm scikit-learn pandas

In [ ]:
import pandas as pd

VERSION = '2cl'

train = pd.read_parquet(os.path.join(TOK_DIR, f'df_train_{VERSION}.parquet'))
test  = pd.read_parquet(os.path.join(TOK_DIR, f'df_test_{VERSION}.parquet'))

print('train:', train.shape, ' test:', test.shape)

## Konfiguracja OpenRouter

Klucz **MUSI** byc w zmiennej srodowiskowej. Na Colabie ustawia sie przez `userdata.get('OPENROUTER_API_KEY')`.

In [ ]:
import os, re, time, getpass
import requests
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

if not os.environ.get('OPENROUTER_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    except Exception:
        os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OPENROUTER_API_KEY: ')

OPENROUTER_API_KEY = os.environ['OPENROUTER_API_KEY']
OPENROUTER_URL = 'https://openrouter.ai/api/v1/chat/completions'

MODELS = [
    'openai/gpt-oss-20b:free',
    'google/gemma-4-31b-it:free',
    'cognitivecomputations/dolphin-mistral-24b-venice-edition:free',
    'meta-llama/llama-3.3-70b-instruct:free',
]

MAX_TEST_SAMPLES = 120
TEMPERATURE = 0
MAX_TOKENS = 8
SLEEP_BETWEEN_CALLS = 0.15

LABEL_INT_TO_STR = {0: 'fake', 1: 'true'}
LABEL_STR_TO_INT = {'fake': 0, 'true': 1}

SYSTEM_PROMPT = (
    'You are a strict binary classifier for misinformation detection. '
    'Return ONLY one token: fake or true.'
)

few_shot_examples = []
for label_int in [0, 1, 0, 1]:
    row = train[train['label'] == label_int].sample(
        1, random_state=42 + len(few_shot_examples)).iloc[0]
    few_shot_examples.append({'statement': row['statement'],
                              'label': LABEL_INT_TO_STR[label_int]})

few_shot_block = '\n'.join([
    f"statement: {e['statement']}\nlabel: {e['label']}" for e in few_shot_examples
])

print('Few-shot examples prepared:', len(few_shot_examples))

In [ ]:
def build_user_prompt(statement, mode='zero'):
    if mode == 'zero':
        return (
            'Task: classify if the statement is fake or true.\n'
            'Return only: fake OR true.\n\n'
            f'statement: {statement}'
        )
    return (
        'Task: classify if the statement is fake or true.\n'
        'Use examples below.\n'
        'Return only: fake OR true.\n\n'
        f'{few_shot_block}\n\n'
        f'statement: {statement}\n'
        'label:'
    )


def parse_label(text):
    t = text.strip().lower()
    m = re.search(r'\b(fake|true)\b', t)
    if not m:
        return None
    return m.group(1)


def call_openrouter(model, user_prompt, retries=3):
    headers = {
        'Authorization': f'Bearer {OPENROUTER_API_KEY}',
        'Content-Type': 'application/json',
    }
    payload = {
        'model': model,
        'temperature': TEMPERATURE,
        'max_tokens': MAX_TOKENS,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
    }
    for attempt in range(retries):
        try:
            resp = requests.post(OPENROUTER_URL, headers=headers,
                                 json=payload, timeout=45)
            if resp.status_code == 200:
                return resp.json()['choices'][0]['message']['content']
            if resp.status_code in [429, 500, 502, 503, 504] and attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
                continue
            return f'ERROR_STATUS_{resp.status_code}'
        except requests.RequestException:
            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
                continue
            return 'ERROR_REQUEST'


def evaluate_model_llm(model, mode='zero', n_samples=MAX_TEST_SAMPLES):
    df = test.sample(n=min(n_samples, len(test)), random_state=42).reset_index(drop=True)
    y_true, y_pred = [], []
    invalid = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'{model} | {mode}'):
        prompt = build_user_prompt(row['statement'], mode=mode)
        raw = call_openrouter(model, prompt)
        lbl = parse_label(raw)
        if lbl is None:
            invalid += 1
            lbl = 'fake'
        y_true.append(row['label'])
        y_pred.append(LABEL_STR_TO_INT[lbl])
        time.sleep(SLEEP_BETWEEN_CALLS)

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred,
                                                   average='macro', zero_division=0)
    print(f'\n=== {model} | {mode} ===  acc={acc:.3f}  macro-F1={f1:.3f}  unparsed={invalid}')
    print(classification_report(y_true, y_pred, zero_division=0))
    return {
        'model': model, 'mode': mode, 'samples': len(df),
        'invalid_outputs': invalid,
        'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1,
    }

In [ ]:
llm_results = []
for model in MODELS:
    llm_results.append(evaluate_model_llm(model, mode='zero',
                                           n_samples=MAX_TEST_SAMPLES))
    llm_results.append(evaluate_model_llm(model, mode='few',
                                           n_samples=MAX_TEST_SAMPLES))

llm_results_df = pd.DataFrame(llm_results)
llm_results_df

In [ ]:
ranking_llm = (llm_results_df
               .sort_values(['f1', 'accuracy'], ascending=False)
               .reset_index(drop=True)
               .round(4))
print('Ranking (LLM zero-shot / few-shot):')
ranking_llm